# CSCI 6379 / 4353 &mdash; Topic 4
## Getting Data and Splitting It

Dr. Dongchul Kim &middot; Department of Computer Science, UTRGV &middot; Fall 2026

---

This notebook is the runnable companion to the Topic 4 note. Everything in sections 1 to 6
also runs in the browser on the course page; **section 7 (PyTorch) does not**, because the
browser runtime has no PyTorch. That is what this notebook is for.

Run cells with **Shift + Enter**.


## 0. What we are working with


In [ ]:
import sys, numpy as np, pandas as pd, sklearn
print('python      :', sys.version.split()[0])
print('numpy       :', np.__version__)
print('pandas      :', pd.__version__)
print('scikit-learn:', sklearn.__version__)


## 1. Loading a bundled dataset

`load_iris()` returns a `Bunch`: a dictionary whose keys also work as attributes.


In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
print(type(iris))
print(sorted(iris.keys()))


In [ ]:
print('data shape  :', iris.data.shape)
print('target shape:', iris.target.shape)
print('features    :', iris.feature_names)
print('classes     :', iris.target_names)


`iris.data` is the $X$ from Topic 3, `iris.target` is the $y$.

The same loader gives you a **regression** dataset. Look at `target`: numbers, not labels.


In [ ]:
from sklearn.datasets import load_diabetes

d = load_diabetes()
print('data shape   :', d.data.shape)
print('target shape :', d.target.shape)
print('features     :', d.feature_names)
print('first targets:', d.target[:5])


## 2. Print before you model

Loading a dataset is not the same as knowing what is in it.


In [ ]:
print(iris.data[:5])
print(iris.target[:5])
print(iris.target[48:53])   # class 1 starts at row 50: the file is SORTED


In [ ]:
df = pd.DataFrame(iris.data, columns=['sepal_len', 'sepal_wid', 'petal_len', 'petal_wid'])
df['species'] = iris.target_names[iris.target]
print(df.head())


`head()` is misleading here, because the file is sorted by species. `sample()` is fairer.


In [ ]:
print(df.sample(5, random_state=0))


Two summaries worth running on **every** new dataset: is it balanced, and what ranges are the features in?


In [ ]:
print(df['species'].value_counts())
print()
print(df.describe().round(2))


## 3. One table, four arrays

`train_test_split` returns **four** things, in the order `X_train, X_test, y_train, y_test`.
Both `X` pieces come back before both `y` pieces. This order trips up almost everyone once.


In [ ]:
from sklearn.model_selection import train_test_split

X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print('X_train:', X_train.shape, '  y_train:', y_train.shape)
print('X_test :', X_test.shape,  '  y_test :', y_test.shape)


Sanity checks to run every time:

- `X_train` and `y_train` share the same first number. They must.
- `X` is 2-D, `y` is 1-D. The comma in `(120,)` is not a typo.
- $120 + 30 = 150$: nothing created, nothing lost.


## 4. `random_state` and `stratify`

`train_test_split` shuffles before it cuts. Without a seed you get a different split, and a
different score, on every run.


In [ ]:
a = train_test_split(X, y, test_size=0.2, random_state=0)[2][:5]
b = train_test_split(X, y, test_size=0.2, random_state=0)[2][:5]
print('seed 0, run 1:', a)
print('seed 0, run 2:', b)


`stratify=y` keeps each class's share the same in both halves.


In [ ]:
_, _, _, y_test_strat = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
_, _, _, y_test_plain = train_test_split(X, y, test_size=0.2, random_state=42)

print('with stratify   :', np.bincount(y_test_strat))
print('without stratify:', np.bincount(y_test_plain))


On balanced Iris the damage is small. On imbalanced data an unstratified test set can end up
with **zero** examples of the rare class, and then recall is undefined.


## 5. A three-way split is two calls

The second `test_size` is a fraction of **what is left**, not of the original.


In [ ]:
X_tmp, X_test3, y_tmp, y_test3 = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y)

X_train3, X_val3, y_train3, y_val3 = train_test_split(
    X_tmp, y_tmp, test_size=0.1732, random_state=42, stratify=y_tmp)

print('train:', len(X_train3), ' val:', len(X_val3), ' test:', len(X_test3))


## 6. Fit the scaler on train only

Fit on train. Transform everything. Never the other way round.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)      # learns mean and std from TRAIN only
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)        # test is transformed, never fitted

print('train mean before:', X_train.mean(axis=0).round(2))
print('train mean after :', X_train_s.mean(axis=0).round(2))
print('test  mean after :', X_test_s.mean(axis=0).round(2))


The training mean is zero: precisely the data the scaler measured. (Those minus signs are
negative zero; `-0.0 == 0.0` is `True`.) The test mean is near zero but clearly not zero, and
that gap is the honest signal that the test set was never consulted.

**If the test mean comes out at zero too, you leaked.** Nothing crashes. The score is just too good.


Try it yourself: the wrong way, so you can see the difference.


In [ ]:
# WRONG on purpose: fitting on everything
bad = StandardScaler().fit(X)               # <- the test rows are in here
print('test mean, leaked scaler:', bad.transform(X_test).mean(axis=0).round(2))
print('test mean, honest scaler:', X_test_s.mean(axis=0).round(2))


---
# 7. The same job in PyTorch

**This is the part that cannot run in the browser.** Colab has PyTorch preinstalled.


In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split
print('torch:', torch.__version__)
print('cuda :', torch.cuda.is_available())


`TensorDataset` wraps arrays you already have; `DataLoader` hands the model mini-batches.


In [ ]:
X_t = torch.tensor(X, dtype=torch.float32)      # (150, 4)
y_t = torch.tensor(y, dtype=torch.long)         # (150,)
full = TensorDataset(X_t, y_t)

g = torch.Generator().manual_seed(42)           # the random_state equivalent
train_set, val_set, test_set = random_split(full, [105, 22, 23], generator=g)

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=16)
test_loader  = DataLoader(test_set,  batch_size=16)

print(len(train_set), len(val_set), len(test_set))

xb, yb = next(iter(train_loader))
print(xb.shape, yb.shape)                       # torch.Size([16, 4]) torch.Size([16])


Three differences from scikit-learn worth remembering:

- `random_split` takes **counts**, not fractions, and splits into as many pieces as you ask in one call.
- It has **no `stratify`**. For imbalanced classification, split with scikit-learn first, then wrap in tensors.
- `shuffle=True` belongs on the **training** loader only.


Watch the loader walk the whole training set, one batch at a time. The last batch is short:
105 is not a multiple of 16.


In [ ]:
for i, (xb, yb) in enumerate(train_loader):
    print(f'batch {i}: X {tuple(xb.shape)}  y {tuple(yb.shape)}')


## 8. Image datasets: torchvision

torchvision downloads and caches for you. `transform` converts each raw image to a tensor as it is read.


In [ ]:
from torchvision import datasets, transforms

tf = transforms.ToTensor()
train_mnist = datasets.MNIST(root='./data', train=True,  download=True, transform=tf)
test_mnist  = datasets.MNIST(root='./data', train=False, download=True, transform=tf)

print(len(train_mnist), len(test_mnist))        # 60000 10000
img, label = train_mnist[0]
print(img.shape, label)                         # torch.Size([1, 28, 28]) 5


MNIST arrives **already split**: a fixed 60,000 / 10,000 division, so every paper reporting an
MNIST number reports it on the same 10,000 images. When a dataset ships with an official split, use it.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, i in zip(axes, range(8)):
    im, lab = train_mnist[i]
    ax.imshow(im.squeeze(), cmap='gray')
    ax.set_title(str(lab))
    ax.axis('off')
plt.tight_layout()
plt.show()


---
## Your turn

1. Load `load_wine()` from `sklearn.datasets`. How many samples, features, and classes? Is it balanced?
2. Split it 70 / 15 / 15 with `stratify`, and print the three lengths.
3. Print the class counts of your test set with and without `stratify`. Which one is worse, and why?
4. Wrap the wine training set in a `TensorDataset` and a `DataLoader` with `batch_size=32`.
   How many batches do you get, and what shape is the last one?


In [ ]:
# your work here
